In [1]:
import numpy as np
import pandas as pd
df=pd.read_csv('spotify_millsongdata.csv')
df.head()

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [2]:
df=df.sample(10000).drop('link', axis=1).reset_index(drop=True)
df.head()

,artist,song,text
0,Orphaned Land,The Path Part 1 - Treading Through Darkness,"Darkness, I believe thee not \r\nThy empty wo..."
1,Offspring,Way Down The Line,Nothing changes cause it's all the same \r\nT...
2,Backstreet Boys,All Of Your Life(Need Love),"Hey \r\nYeah, yeah \r\nI wanna know \r\nCan..."
3,Matt Redman,Knocking On The Door Of Heaven Lyrics,We will give ourselves no rest \r\n'Till Your...
4,Ofra Haza,Galbi,Ahhh Galbi \r\nGalbi ya heb il hawa \r\nLa t...


In [3]:
df.shape

(10000, 3)

In [4]:
df.isnull().sum()

artist    0
song      0
text      0
dtype: int64

In [5]:
df['text'][0]

"Darkness, I believe thee not  \r\nThy empty words shall avail thee naught  \r\nA fire in this heart of mine  \r\nTo gaze again upon these walls of thine  \r\nDesire to soar once more  \r\nUpon these broken wings on which I have flown before  \r\nTongues of flame shall paint the canvas red  \r\nAs once told, I shall part the rising sea  \r\nSeeds from the blood that I she'd  \r\nFeet sink deeper into grains of golden sand knee deep  \r\nEvery step I take is a drop in this sea of sleep  \r\nIn which I have swam and drowned  \r\nThe wind whispers death, as temptation drips from her song  \r\nTears run dry - will I survive?  \r\nHear my cry - will I arrive?  \r\nHeading home forever more  \r\nAll past grief is now gone  \r\nThe gift of life to me they bequest  \r\nMine is the sight in the blindness  \r\nAs I'm treading through the path, in darkness\r\n\r\n"

In [6]:
# Data Preprocessing
df['text']=df['text'].str.lower().replace(r'^\w\s',' ').replace(r'\n',' ', regex=True)
df.head()

,artist,song,text
0,Orphaned Land,The Path Part 1 - Treading Through Darkness,"darkness, i believe thee not \r thy empty wor..."
1,Offspring,Way Down The Line,nothing changes cause it's all the same \r th...
2,Backstreet Boys,All Of Your Life(Need Love),"hey \r yeah, yeah \r i wanna know \r can we..."
3,Matt Redman,Knocking On The Door Of Heaven Lyrics,we will give ourselves no rest \r 'till your ...
4,Ofra Haza,Galbi,ahhh galbi \r galbi ya heb il hawa \r la tin...


In [7]:
df['text'][0]

"darkness, i believe thee not  \r thy empty words shall avail thee naught  \r a fire in this heart of mine  \r to gaze again upon these walls of thine  \r desire to soar once more  \r upon these broken wings on which i have flown before  \r tongues of flame shall paint the canvas red  \r as once told, i shall part the rising sea  \r seeds from the blood that i she'd  \r feet sink deeper into grains of golden sand knee deep  \r every step i take is a drop in this sea of sleep  \r in which i have swam and drowned  \r the wind whispers death, as temptation drips from her song  \r tears run dry - will i survive?  \r hear my cry - will i arrive?  \r heading home forever more  \r all past grief is now gone  \r the gift of life to me they bequest  \r mine is the sight in the blindness  \r as i'm treading through the path, in darkness\r \r "

In [8]:
import nltk
from nltk.stem.porter import PorterStemmer
stemmer=PorterStemmer()

In [9]:
def token(txt):
    token=nltk.word_tokenize(txt)
    a=[stemmer.stem(w) for w in token]
    return " ".join(a)

In [10]:
df['text'].apply(lambda x: token)

0       <function token at 0x000002649FD8A0C0>
1       <function token at 0x000002649FD8A0C0>
2       <function token at 0x000002649FD8A0C0>
3       <function token at 0x000002649FD8A0C0>
4       <function token at 0x000002649FD8A0C0>
                         ...                  
9995    <function token at 0x000002649FD8A0C0>
9996    <function token at 0x000002649FD8A0C0>
9997    <function token at 0x000002649FD8A0C0>
9998    <function token at 0x000002649FD8A0C0>
9999    <function token at 0x000002649FD8A0C0>
Name: text, Length: 10000, dtype: object

In [11]:
# Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
tfid=TfidfVectorizer(analyzer='word', stop_words='english')
matrix=tfid.fit_transform(df['text']) # convert lyrics from songs into numerical vectors

In [12]:
similar=cosine_similarity(matrix) # compares each song vector with every other song

In [13]:
similar[0]

array([1.        , 0.00539738, 0.02115625, ..., 0.00988444, 0.        ,
       0.00933204])

In [14]:
# Recommender Function
def recommender(song_name):
    idx=df[df['song']==song_name].index[0]
    distance=sorted(list(enumerate(similar[idx])), reverse=True, key = lambda x:x[1])
    song=[]
    for s_id in distance[1:6]:
        song.append(df.iloc[s_id[0]].song)
    return song

In [15]:
recommender("Sin City")

['Warsaw', 'Fire', 'Rain Of Fire', "It Won't Cool Off", 'You Move Me']

In [16]:
import pickle
pickle.dump(similar, open("english_similarity","wb"))
pickle.dump(df, open("english_df","wb"))